# 20 — Paper 3: PM2.5 รายเดือนกับการเจ็บป่วยเฉียบพลันระดับตำบล (Bayesian NB-BYM2)

**คำถาม:** เมื่อ PM2.5 ของตำบลสูงขึ้น จำนวนผู้ป่วย 11 โรคที่เกี่ยวกับ PM2.5 (HDC) ในเดือนเดียวกัน/เดือนถัดไปเพิ่มขึ้นเท่าไร
**ชนิดแบบจำลอง:** แบบจำลองหาความสัมพันธ์ (association) — ประมาณ **IRR ต่อ PM2.5 ที่เพิ่มขึ้น 10 µg/m³** ไม่ใช่แบบจำลองทำนายจำนวนผู้ป่วย

| ส่วน | เนื้อหา |
|---|---|
| ข้อมูล | ตำบล × เดือน ปีงบ 2565–2569 (ต.ค. 2564 – ก.ย. 2569) · PM2.5 จาก Notebook 06 · ผู้ป่วยรายเดือนจาก HDC (`s_pm25_disease`) · ประชากร WorldPop 2020 |
| การสัมผัส | PM2.5 ถ่วงประชากร **แยกเป็น 2 ส่วน**: ส่วนที่เปลี่ยนตามเวลาภายในตำบล (*within* — ผลหลัก) และค่าเฉลี่ยระยะยาวของตำบล (*between*) |
| เวลา | ผลของเดือนในรอบปี (ฤดูกาล) + ผลของปีงบประมาณ (หรือผลรายเดือนทุกเดือนในการวิเคราะห์ความไว) |
| พื้นที่ | ผลสุ่ม BYM2 (Riebler et al., 2016) บนโครงข่ายเพื่อนบ้าน Queen |
| ความไว | NB แบบ fixed effect รายตำบล (เร็ว) ภายใต้ทางเลือกต่าง ๆ: lag, ที่มาของค่า PM2.5, กลุ่มโรค, ตัดตำบลผิดปกติ, การควบคุมเวลา |

**ทำไมต้องแยก within/between:** ค่าเฉลี่ย PM2.5 ระหว่างตำบลต่างกันเพียง ~5 µg/m³ และลำดับตำบลแทบไม่เปลี่ยนข้ามปี (Notebook 06: Spearman ρ ระหว่างปี ≈ 0.89) ความต่างระหว่างตำบลจึงปนกับปัจจัยอื่นของพื้นที่ (การเข้าถึงบริการ ที่ตั้งโรงพยาบาล ความเป็นเมือง) และกับผลสุ่มเชิงพื้นที่ (*spatial confounding*; Hodges & Reich, 2010) ส่วน *within* เปรียบเทียบตำบลกับตัวเองในเวลาต่างกัน จึงตัดปัจจัยที่คงที่ของตำบลออกโดยอัตโนมัติ (Begg & Parides, 2003)

> **ข้อมูลผู้ป่วยรายปี/รายเดือน:** ถ้าไฟล์ HDC เป็นรายปี (`s_pm25_disease` ของ Open Data — คอลัมน์ `year` คือ **ปีปฏิทิน พ.ศ.** ม.ค.–ธ.ค. ตรวจแล้วจากรายงานรายเดือนของ HDC) notebook จะวิเคราะห์ **ตำบล × ปี** ให้อัตโนมัติ (ส่วน *within* มีเพียง 5 ค่าต่อตำบล อำนาจทดสอบจึงต่ำ) เมื่อได้ไฟล์รายเดือนให้เปลี่ยน `HDC_MONTHLY_CSV` แล้วรันใหม่
>
> `Runtime → Run all` · ส่วน Bayesian ใช้เวลา ~5–30 นาที (เร็วขึ้นมากเมื่อใช้ NumPyro) — ตั้ง `FAST = True` เพื่อลองรันก่อน

## 0. CONFIG

In [ ]:
# ===== Repository data (data/processed/) =====
# Opened from GitHub in Colab: the repository is cloned so that data/processed/ is available.
# Run locally: start Jupyter in the repository (or its notebooks/ folder).
# If data/processed/ is not found, the notebook falls back to the original Google Drive paths.
import os, subprocess
REPO_URL = "https://github.com/songwutsa/paper3-pm25-acute-illness-saraburi"
USE_REPO = True          # False = use the original Google Drive paths only
REPO_DIR = None
for _d in (os.getcwd(), os.path.dirname(os.getcwd()), "/content/paper3-pm25-acute-illness-saraburi"):
    if os.path.isdir(os.path.join(_d, "data", "processed")):
        REPO_DIR = _d; break
if USE_REPO and REPO_DIR is None and os.path.isdir("/content"):
    REPO_DIR = "/content/paper3-pm25-acute-illness-saraburi"
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR], check=False)
REPO_DATA = os.path.join(REPO_DIR, "data", "processed") if REPO_DIR else ""
USE_REPO_DATA = bool(USE_REPO and REPO_DATA and os.path.isdir(REPO_DATA))

def repo_file(name):
    """Path of a file in data/processed/ (matched by file name), or None."""
    p = os.path.join(REPO_DATA, os.path.basename(str(name))) if USE_REPO_DATA else ""
    return p if p and os.path.exists(p) else None

print("Repository data:", REPO_DATA if USE_REPO_DATA else "not found - using Google Drive paths")

In [ ]:
# ---- ไฟล์จาก Notebook 06 (OUTDIR = MyDrive/PM25_Saraburi_notebooks) ----
DATA_DIR = "/content/drive/MyDrive/PM25_Saraburi_notebooks"
PM_PANEL_CSV = "pm25_tambon_monthly_fy2565_2569.csv"     # ตำบล × เดือน ต.ค. 2564 – ก.ย. 2569
SAT_MONTHLY_CSV = "sat_pm25_tambon_monthly.csv"          # ดาวเทียมรายเดือน 2016–2024 (ใช้เติม lag ของเดือนแรก)
BOUNDARY_GPKG = "sat_pm25_tambon_summary.gpkg"           # ขอบเขต 111 ตำบล + รหัส + ประชากร WorldPop

# ---- ข้อมูลผู้ป่วย HDC (CSV/XLSX) — path ใน Drive (เทียบกับ MyDrive หรือเป็น path เต็ม) ----
# ไฟล์ที่มีอยู่ตอนนี้เป็น "รายปี" (MOPH Open Data s_pm25_disease, year = ปีปฏิทิน พ.ศ. 2565–2569; ปี 2569 มีถึง ก.ย.) → รวม PM2.5 เป็นรายปีอัตโนมัติ
# เมื่อได้ไฟล์ "รายเดือน" (มีคอลัมน์เดือน) ให้เปลี่ยน path — notebook จะวิเคราะห์รายเดือนอัตโนมัติ
HDC_MONTHLY_CSV = "000_research/11HealthImpactAssessment_SW/0data/hdc_saraburi_pm25_disease_tambon_monthly_2565_2569.csv"
HDC_YEAR_IS_FISCAL = False           # (ไฟล์รายเดือน) True ถ้าคอลัมน์ปีเป็น "ปีงบประมาณ" (ต.ค.–ธ.ค. จะเลื่อนเป็นปีปฏิทินก่อนหน้า)
DROP_INCOMPLETE_MONTHS = True        # ตัดเดือนที่ HDC บันทึกไม่ครบ (คอลัมน์ flag_incomplete = ธ.ค. ทุกปี: ตาราง HDC หยุดปรับปรุงต้น ม.ค.)
MIN_MONTHS_FY = 10                   # (ไฟล์รายปี) จำนวนเดือนที่มีค่า PM2.5 ขั้นต่ำในปีงบ

# ---- การสัมผัส ----
EXPOSURE_MODE = "satscale"           # "satscale" = ดาวเทียม แล้วต่อด้วยสมการ (15) สเกลดาวเทียม (หลัก ไม่มีรอยต่อ)
                                     # "proposal" = คอลัมน์ pm25 ของ Notebook 06 (สมการ 15 สเกลสถานี ตามโครงร่าง)
                                     # "satellite_only" = เฉพาะเดือนที่มีดาวเทียม (ต.ค. 2564 – ธ.ค. 2567)
LAG = 0                              # 0 = เดือนเดียวกัน, 1 = เดือนก่อนหน้า
PM_UNIT = 10                         # รายงาน IRR ต่อ 10 µg/m³

# ---- ตัวแปรตาม ----
OUTCOME = "total"                    # "total" (11 โรค) | "respiratory" (7 โรค) | "cardiovascular" (2 โรค) | "eye_skin" | หรือชื่อโรค เช่น "COPD"

# ---- การควบคุมเวลา ----
TIME_CONTROL = "moy+fy"              # "moy+fy" = เดือนในรอบปี + ปีงบประมาณ (หลัก) | "month_fe" = ผลของทุกเดือน (เข้มที่สุด)

# ---- ตัวแปรระดับตำบลเพิ่มเติม (ไม่บังคับ) เช่น สัดส่วนอุตสาหกรรม/ความหนาแน่นถนน/ประชากรริมถนน ----
COVARIATE_FILES = []                 # รายการไฟล์ CSV/GPKG ใน Drive ที่มีคอลัมน์ a_code
COVARIATES = []                      # ชื่อคอลัมน์ที่ใช้ เช่น ["industry_pct", "road_density", "pop_density"]

# ---- ตำบลที่อัตราป่วยสูงผิดปกติ (อาจเป็นผลจากที่ตั้งหน่วยบริการ) ----
OUTLIER_RATIO = 3.0                  # อัตราตำบล / อัตราจังหวัด ≥ ค่านี้ = ผิดปกติ
EXCLUDE_OUTLIERS = False             # True = ตัดออกจากแบบจำลองหลัก (ผลทั้งสองแบบมีในตารางความไวอยู่แล้ว)

# ---- MCMC ----
FAST = False                         # True = ลองรันเร็ว (ห้ามใช้รายงานผล)
DRAWS, TUNE, CHAINS = (400, 600, 2) if FAST else (1000, 1500, 4)
SAMPLER = "auto"                     # "auto" (numpyro ถ้ามี) | "numpyro" | "pymc"
SEED = 2026
CONF = 0.95
SAVE_TO_DRIVE = True

In [ ]:
# ===== Use data/processed/ from the repository when available (the Drive paths above are the fallback) =====
if USE_REPO_DATA:
    DATA_DIR = REPO_DATA
    for _v in ("PM_PANEL_CSV", "SAT_MONTHLY_CSV", "BOUNDARY_GPKG", "HDC_MONTHLY_CSV", "DIW_CSV", "ROADSIDE_CSV"):
        if globals().get(_v): globals()[_v] = os.path.basename(globals()[_v])
    if "ROADS_CANDIDATES" in globals():
        ROADS_GEOJSON = "osm_highway_1_2.geojson"; ROADS_CANDIDATES = [ROADS_GEOJSON]
    if "ERA5_CACHE_DIR" in globals():
        ERA5_CACHE_DIR = os.path.join(REPO_DATA, "ERA5_cache")   # hourly ERA5 2022–2024 (Open-Meteo archive); other years are downloaded
    SAVE_TO_DRIVE = False                                  # outputs go to the working folder, not data/processed/
    print("Using repository data in", DATA_DIR)

In [ ]:
import importlib.util, subprocess, sys
need = {"libpysal": "libpysal", "esda": "esda", "pymc": "pymc", "arviz": "arviz", "numpyro": "numpyro", "openpyxl": "openpyxl", "netCDF4": "netCDF4"}
missing = [p for m, p in need.items() if importlib.util.find_spec(m) is None]
if missing:
    print("ติดตั้ง:", missing); subprocess.run(f"{sys.executable} -m pip -q install " + " ".join(missing), shell=True)

In [ ]:
import os, re, json, time, warnings, subprocess, importlib.util
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt, requests
from scipy import sparse
from scipy.sparse.csgraph import connected_components
import statsmodels.api as sm, statsmodels.formula.api as smf
import libpysal, esda
from libpysal.weights import Queen
import pymc as pm, arviz as az, pytensor.tensor as pt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
UTM = "EPSG:32647"

def setup_thai_font():
    import matplotlib.font_manager as fm, glob
    base = "/content" if os.path.isdir("/content") else "."
    path = os.path.join(base, "Sarabun-Regular.ttf")
    if not os.path.exists(path):
        try:
            r = requests.get("https://github.com/google/fonts/raw/main/ofl/sarabun/Sarabun-Regular.ttf", timeout=60)
            if r.status_code == 200 and len(r.content) > 50000: open(path, "wb").write(r.content)
        except Exception:
            pass
    if os.path.exists(path):
        fm.fontManager.addfont(path)
        plt.rcParams["font.family"] = ["DejaVu Sans", fm.FontProperties(fname=path).get_name()]
setup_thai_font()

IN_COLAB = os.path.isdir("/content")
if IN_COLAB and DATA_DIR.startswith("/content/drive"):
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"): drive.mount("/content/drive")
DATA_DIR = os.environ.get("PAPER3_DATA_DIR", DATA_DIR)          # (ใช้ทดสอบนอก Colab)
HDC_MONTHLY_CSV = os.environ.get("PAPER3_HDC", HDC_MONTHLY_CSV)
OUTDIR = os.path.join(DATA_DIR, "paper3") if SAVE_TO_DRIVE else "./paper3"
os.makedirs(OUTDIR, exist_ok=True)

def _path(p):
    if not p: return p
    if os.path.isabs(p): return p
    for base in (DATA_DIR, "/content/drive/MyDrive", "."):
        if os.path.exists(os.path.join(base, p)): return os.path.join(base, p)
    return os.path.join(DATA_DIR, p)
print("PyMC", pm.__version__, "| ArviZ", az.__version__, "| OUTDIR =", OUTDIR)

## 1. PM2.5 รายตำบลรายเดือนและขอบเขตตำบล (จาก Notebook 06)

**ที่มาของค่า PM2.5 (`EXPOSURE_MODE`)**
- ต.ค. 2564 – ธ.ค. 2567: ดาวเทียม SatPM2.5 V6.GL.03 ถ่วงประชากร (เหมือนกันทุกแบบ)
- ม.ค. 2568 – ก.ย. 2569: สมการที่ (15) — ค่าสถานี 24T/25T × สัดส่วนรายฤดูของตำบล
  - `satscale` หารด้วยค่าเฉลี่ยรายฤดูที่สถานีวัดได้ → อยู่บนสเกลดาวเทียม ต่อเนื่องกับช่วงก่อนหน้า (ทดสอบย้อนหลังใน Notebook 06: RMSE 5.2 µg/m³)
  - `proposal` หารด้วยค่าดาวเทียมที่กริดสถานี (ตามโครงร่าง) → อยู่บนสเกลสถานี ค่า **กระโดดที่ ม.ค. 2568** (ตำบลที่อ้างอิง 25T ลดลง ตำบลที่อ้างอิง 24T เพิ่มขึ้น; RMSE 7.3 µg/m³)

ตารางท้ายเซลล์แสดงค่าเฉลี่ยรายปีงบแยกตามสถานีอ้างอิง — ถ้าแบบใดค่าเปลี่ยนแบบขั้นบันไดที่ปีงบ 2568 แสดงว่ามีรอยต่อ

In [ ]:
for f in [PM_PANEL_CSV, BOUNDARY_GPKG]:
    if not os.path.exists(_path(f)):
        raise FileNotFoundError(f"ไม่พบ {_path(f)} — รัน Notebook 06 (SAVE_TO_DRIVE=True) ก่อน")
def _code6(s):
    return s.astype(str).str.replace(r"\.0$", "", regex=True).str.replace(r"\D", "", regex=True).str[:6]

PM = pd.read_csv(_path(PM_PANEL_CSV), encoding="utf-8-sig", dtype={"a_code": str})
PM["a_code"] = _code6(PM["a_code"])
BND = gpd.read_file(_path(BOUNDARY_GPKG))
BND["a_code"] = _code6(BND["a_code"])
BND = BND[["a_code", "name", "pop", "area_km2", "geometry"]].drop_duplicates("a_code").reset_index(drop=True)
print(f"✓ PM2.5 panel: {PM.a_code.nunique()} ตำบล × {PM.ym.nunique()} เดือน ({PM.ym.min()} ถึง {PM.ym.max()})")
print(f"✓ ขอบเขต: {len(BND)} ตำบล, ประชากรรวม {BND['pop'].sum():,.0f} คน (WorldPop 2020)")

# ค่า PM2.5 เดือนก่อนหน้าเดือนแรก (ก.ย. 2564) สำหรับ lag — ดาวเทียมจาก sat_pm25_tambon_monthly.csv
PREV = None
if os.path.exists(_path(SAT_MONTHLY_CSV)):
    S_ = pd.read_csv(_path(SAT_MONTHLY_CSV), encoding="utf-8-sig", dtype={"a_code": str})
    S_["a_code"] = _code6(S_["a_code"]); S_["ym"] = S_["time"].astype(str).str[:7]
    PREV = S_[["a_code", "ym", "pm25_pop"]]

def exposure_series(mode):
    """คืน DataFrame (a_code, ym, pm, pm_src) ตามแบบที่เลือก"""
    E = PM[["a_code", "ym", "nearest_station", "fy_be"]].copy()
    sat = PM["pm25_sat_pop"]
    if mode == "satscale":
        E["pm"] = sat.where(sat.notna(), PM["pm25_eq15_satscale"])
        E["pm_src"] = np.where(sat.notna(), "satellite", np.where(PM["pm25_eq15_satscale"].notna(), "eq15_satscale", "missing"))
    elif mode == "proposal":
        E["pm"] = PM["pm25"]; E["pm_src"] = PM["pm25_source"].fillna("missing")
    elif mode == "satellite_only":
        E["pm"] = sat; E["pm_src"] = np.where(sat.notna(), "satellite", "missing")
    else:
        raise ValueError(mode)
    return E

def add_lag(E, lag):
    E = E.copy()
    if lag == 0:
        E["pm_use"] = E["pm"]; return E
    full = E[["a_code", "ym", "pm"]]
    if PREV is not None:
        first = pd.Period(E.ym.min(), "M")
        pre = PREV[PREV.ym.isin([str(first - k) for k in range(1, lag + 1)])].rename(columns={"pm25_pop": "pm"})
        full = pd.concat([pre, full], ignore_index=True)
    full = full.assign(per=pd.PeriodIndex(full.ym, freq="M")).drop_duplicates(["a_code", "ym"])
    shifted = full.assign(per=full.per + lag)[["a_code", "per", "pm"]].rename(columns={"pm": "pm_use"})
    E["per"] = pd.PeriodIndex(E.ym, freq="M")
    return E.merge(shifted, on=["a_code", "per"], how="left").drop(columns="per")

E_MAIN = exposure_series(EXPOSURE_MODE)
print(f"\nEXPOSURE_MODE = {EXPOSURE_MODE}: ที่มาของค่า (ตำบล-เดือน) {E_MAIN.pm_src.value_counts().to_dict()}")
miss_m = sorted(E_MAIN.loc[E_MAIN.pm.isna(), "ym"].unique())
if miss_m: print(f"⚠ เดือนที่ไม่มีค่า PM2.5 (จะถูกตัดออก): {', '.join(miss_m)}")

cmp_ = pd.concat({m: exposure_series(m).groupby(["fy_be", "nearest_station"])["pm"].mean().unstack()
                  for m in ["satscale", "proposal"]}, axis=1).round(1)
print("\nค่าเฉลี่ย PM2.5 (µg/m³) รายปีงบ แยกตามสถานีอ้างอิงของตำบล — ตรวจรอยต่อที่ปีงบ 2568:")
display(cmp_)

## 2. ข้อมูลผู้ป่วย HDC รายเดือน

ฟังก์ชันอ่านไฟล์รองรับรูปแบบที่พบบ่อย:
- **รหัสตำบล:** คอลัมน์ `a_code` / `areacode` / `tambon_code` … (รหัส 8 หลักจะตัดเหลือ 6 หลัก = จังหวัด-อำเภอ-ตำบล)
- **เวลา:** `ym` (เช่น `2022-01`, `202201`, `2565-01`) หรือ `year` + `month` หรือ `date` — ปี พ.ศ. แปลงเป็น ค.ศ. อัตโนมัติ
- **โรค:** คอลัมน์ชื่อโรค 11 โรค หรือรหัสคอลัมน์ดิบ `r1_1 … r4` หรือรูปแบบยาว (`disease`, `cases`)

ตรวจผลลัพธ์บรรทัด "ช่วงเวลา" ให้ตรงกับ 2021-10 ถึง 2026-09 — ถ้าเลื่อนไป 1 ปีในเดือน ต.ค.–ธ.ค. ให้ตั้ง `HDC_YEAR_IS_FISCAL`

In [ ]:
COLMAP = {"r1_1": "COPD", "r1_2": "Asthma", "r1_3": "Pneumonia", "r1_4": "Influenza", "r1_5": "Acute pharyngitis",
          "r1_6": "Chronic rhinitis", "r1_7": "Bronchitis", "r2_1": "Ischemic heart diseases",
          "r2_2": "Cerebrovascular diseases (Stroke)", "r3": "กลุ่มโรคตาอักเสบ", "r4": "กลุ่มโรคผิวหนังอักเสบ"}
DISEASES = list(COLMAP.values())
GROUPS = {"total": DISEASES, "respiratory": DISEASES[:7], "cardiovascular": DISEASES[7:9], "eye_skin": DISEASES[9:11]}
_norm = lambda s: re.sub(r"[\s_\-().]+", "", str(s)).lower()

def _find(cols, names):
    nc = {_norm(c): c for c in cols}
    return next((nc[_norm(n)] for n in names if _norm(n) in nc), None)

def _parse_ym(v):
    d = re.sub(r"\D", "", str(v))
    if len(d) < 5: return None
    y, m = int(d[:4]), int(d[4:6]) if len(d) >= 6 else int(d[4:])
    if y > 2400: y -= 543
    return f"{y:04d}-{m:02d}" if 1 <= m <= 12 else None

def load_hdc_monthly(path):
    df = pd.read_excel(path) if path.lower().endswith((".xlsx", ".xls")) else pd.read_csv(path, encoding="utf-8-sig")
    df.columns = [str(c).strip() for c in df.columns]
    # รูปแบบยาว → กว้าง
    dcol, vcol = _find(df.columns, ["disease", "disease_name", "โรค", "กลุ่มโรค"]), _find(df.columns, ["cases", "count", "n", "value", "จำนวน"])
    # รหัสตำบล
    ccol = _find(df.columns, ["a_code", "areacode", "area_code", "tambon_code", "tmb_code", "subdistrict_code", "tambon_id", "code"])
    if ccol is None:
        for c in df.columns:
            d = df[c].astype(str).str.replace(r"\.0$", "", regex=True)
            if (d.str.fullmatch(r"19\d{4}(\d{2})?")).mean() > 0.8: ccol = c; break
    if ccol is None: raise RuntimeError(f"ไม่พบคอลัมน์รหัสตำบล — คอลัมน์ที่มี: {list(df.columns)}")
    df["a_code"] = _code6(df[ccol])
    # เวลา
    ymc = _find(df.columns, ["ym", "yearmonth", "year_month", "period", "month_year"])
    yc, mc = _find(df.columns, ["year", "yr", "b_year", "byear", "ปี", "fiscal_year", "fy"]), _find(df.columns, ["month", "mon", "m", "เดือน"])
    dc = _find(df.columns, ["date", "date_serv", "วันที่"])
    if ymc is not None:
        df["ym"] = df[ymc].map(_parse_ym)
    elif yc is not None and mc is not None:
        y = pd.to_numeric(df[yc], errors="coerce"); m = pd.to_numeric(df[mc], errors="coerce")
        y = np.where(y > 2400, y - 543, y)
        if HDC_YEAR_IS_FISCAL: y = y - (m >= 10).astype(int)
        df["ym"] = [f"{int(a):04d}-{int(b):02d}" if pd.notna(a) and pd.notna(b) else None for a, b in zip(y, m)]
    elif dc is not None:
        t = pd.to_datetime(df[dc], errors="coerce"); df["ym"] = t.dt.strftime("%Y-%m")
    elif yc is not None:                                   # ไม่มีเดือน → ข้อมูลรายปี (s_pm25_disease: "year" = ปีปฏิทิน พ.ศ. ม.ค.–ธ.ค.)
        y = pd.to_numeric(df[yc], errors="coerce"); y = np.where(y < 2400, y + 543, y)
        df["ym"] = [f"Y{int(a)}" if pd.notna(a) else None for a in y]
    else:
        raise RuntimeError(f"ไม่พบคอลัมน์เวลา (ym / year+month / date) — คอลัมน์ที่มี: {list(df.columns)}")
    df = df[df.ym.notna()]
    if dcol is not None and vcol is not None and not any(_norm(d) in map(_norm, df.columns) for d in DISEASES):
        df[vcol] = pd.to_numeric(df[vcol], errors="coerce").fillna(0)
        df = df.pivot_table(index=["a_code", "ym"], columns=dcol, values=vcol, aggfunc="sum", fill_value=0).reset_index()
    # ชื่อโรค
    ren = {}
    for c in df.columns:
        n = _norm(c)
        hit = next((v for k, v in COLMAP.items() if n == _norm(k)), None) or next((d for d in DISEASES if n == _norm(d)), None)
        if hit: ren[c] = hit
    df = df.rename(columns=ren)
    found = [d for d in DISEASES if d in df.columns]
    if not found: raise RuntimeError(f"ไม่พบคอลัมน์โรค — คอลัมน์ที่มี: {list(df.columns)}")
    if len(found) < len(DISEASES): print(f"⚠ ไม่พบคอลัมน์: {[d for d in DISEASES if d not in found]} — ตั้งเป็น 0")
    for d in DISEASES:
        df[d] = pd.to_numeric(df[d], errors="coerce").fillna(0) if d in df else 0.0
    fcol = _find(df.columns, ["flag_incomplete"])
    df["flag_incomplete"] = pd.to_numeric(df[fcol], errors="coerce").fillna(0).astype(int) if fcol else 0
    return df.groupby(["a_code", "ym"], as_index=False).agg({**{d: "sum" for d in DISEASES}, "flag_incomplete": "max"})

if not HDC_MONTHLY_CSV:
    raise RuntimeError("ตั้ง HDC_MONTHLY_CSV เป็น path ไฟล์ผู้ป่วยรายเดือนใน Drive ก่อน")
HDC = load_hdc_monthly(_path(HDC_MONTHLY_CSV))
HDC = HDC[HDC.a_code.str[:2] == "19"]
TEMPORAL = "annual" if HDC.ym.str.fullmatch(r"Y\d{4}").all() else "monthly"
UNIT = "ปี" if TEMPORAL == "annual" else "เดือน"
if TEMPORAL == "annual":
    TIME_CONTROL = "fy"
    print("ℹ ไฟล์ HDC เป็นข้อมูลรายปี (ปีปฏิทิน พ.ศ.) → วิเคราะห์ ตำบล × ปี (PM2.5 = ค่าเฉลี่ยของเดือนที่มีข้อมูลในปีนั้น)")
    print("  ⚠ ปีล่าสุดอาจยังไม่ครบ 12 เดือน — ตรวจยอดผู้ป่วยรายปีด้านล่าง")
if HDC.flag_incomplete.any():
    fm = sorted(HDC.loc[HDC.flag_incomplete == 1, "ym"].unique())
    print(f"⚠ เดือนที่ HDC บันทึกไม่ครบ ({len(fm)} เดือน: {', '.join(fm)}) → " + ("ตัดออก" if DROP_INCOMPLETE_MONTHS else "ยังใช้อยู่ (DROP_INCOMPLETE_MONTHS=False)"))
    if DROP_INCOMPLETE_MONTHS: HDC = HDC[HDC.flag_incomplete == 0]
print(f"✓ HDC: {HDC.a_code.nunique()} ตำบล, ช่วงเวลา {HDC.ym.min()} ถึง {HDC.ym.max()} ({HDC.ym.nunique()} {UNIT}), "
      f"ผู้ป่วยรวม 11 โรค {HDC[DISEASES].values.sum():,.0f} ครั้ง")
_fy = HDC.ym.str[1:].astype(int) if TEMPORAL == "annual" else pd.to_datetime(HDC.ym).dt.year + 543 + (pd.to_datetime(HDC.ym).dt.month >= 10)
print("ผู้ป่วยรวมรายปี" + ("" if TEMPORAL == "annual" else "งบ") + ":", HDC.assign(fy=_fy).groupby("fy")[DISEASES].sum().sum(axis=1).astype(int).to_dict())
nb_, nh_ = set(BND.a_code), set(HDC.a_code)
print(f"รหัสตำบลตรงกัน {len(nb_ & nh_)} | มีในขอบเขตแต่ไม่มีใน HDC {len(nb_ - nh_)} | มีใน HDC แต่ไม่มีในขอบเขต {len(nh_ - nb_)} {sorted(nh_ - nb_)[:8]}")

## 3. ตารางวิเคราะห์ ตำบล × เดือน และการตรวจข้อมูล

**ตำบลที่อัตราป่วยสูงผิดปกติ:** HDC อาจบันทึกตาม *ที่ตั้งหน่วยบริการ* หรือ *ที่อยู่ผู้ป่วย* — ถ้าเป็นแบบแรก ตำบลที่มีโรงพยาบาลจะมีอัตราสูงเกินจริงหลายเท่า (Notebook 19 พบตำบลที่ความเสี่ยงคงเหลือ 8–19 เท่า) ตารางด้านล่างแสดงตำบลที่อัตรา ≥ `OUTLIER_RATIO` เท่าของจังหวัด
**ต้องตรวจนิยามของ `areacode` ในรายงาน HDC** แล้วระบุในบทที่ 3 · ผลที่ตัด/ไม่ตัดตำบลเหล่านี้อยู่ในตารางความไว (ส่วนที่ 4)

In [ ]:
COV = None
if COVARIATE_FILES and COVARIATES:
    parts = []
    for f in COVARIATE_FILES:
        t = gpd.read_file(_path(f)) if f.lower().endswith((".gpkg", ".geojson", ".shp")) else pd.read_csv(_path(f), encoding="utf-8-sig")
        t = pd.DataFrame(t).drop(columns="geometry", errors="ignore"); t["a_code"] = _code6(t["a_code"])
        parts.append(t[["a_code"] + [c for c in COVARIATES if c in t]].drop_duplicates("a_code").set_index("a_code"))
    COV = pd.concat(parts, axis=1)
    COV = COV.loc[:, ~COV.columns.duplicated()]
    print("✓ ตัวแปรระดับตำบล:", list(COV.columns), f"({COV.notna().all(axis=1).sum()} ตำบลครบ)")

def outcome_values(df, outcome):
    cols = GROUPS.get(outcome, [outcome])
    missing = [c for c in cols if c not in df]
    if missing: raise ValueError(f"ไม่รู้จัก OUTCOME = {outcome}")
    return df[cols].sum(axis=1)

def annualize(E):
    """ค่าเฉลี่ย PM2.5 รายปีงบจากค่ารายเดือน (เฉพาะปีงบที่มีค่า ≥ MIN_MONTHS_FY เดือน)"""
    E = E.assign(fy_be=pd.to_datetime(E.ym).dt.year + 543)          # ปีปฏิทิน พ.ศ. (ชื่อคอลัมน์คงเดิม)
    A = (E.groupby(["a_code", "fy_be"]).agg(pm=("pm", "mean"), n_m=("pm", "count"), nearest_station=("nearest_station", "first"),
                                           n_sat=("pm_src", lambda s_: int((s_ == "satellite").sum()))).reset_index())
    A["pm"] = A.pm.where(A.n_m >= MIN_MONTHS_FY)
    A["pm_src"] = np.where(A.n_sat == A.n_m, "satellite", np.where(A.n_sat == 0, "eq15", "mixed"))
    A["ym"] = "Y" + A.fy_be.astype(int).astype(str)
    return A

def exposure_period(mode, lag):
    E = exposure_series(mode)
    if TEMPORAL == "monthly":
        return add_lag(E, lag)
    A = annualize(E)
    if lag == 0:
        A["pm_use"] = A.pm; return A
    full = A[["a_code", "fy_be", "pm"]]
    if PREV is not None:                                       # ปีงบก่อนหน้าจากดาวเทียม (เช่น ปีงบ 2564)
        P_ = PREV.assign(t=pd.to_datetime(PREV.ym)); P_["fy_be"] = P_.t.dt.year + 543
        pa = P_.groupby(["a_code", "fy_be"]).pm25_pop.agg(["mean", "count"]).reset_index()
        pa = pa[pa["count"] >= MIN_MONTHS_FY].rename(columns={"mean": "pm"})[["a_code", "fy_be", "pm"]]
        full = pd.concat([pa[pa.fy_be < A.fy_be.min()], full], ignore_index=True)
    sh = full.assign(fy_be=full.fy_be + lag).rename(columns={"pm": "pm_use"})
    return A.merge(sh, on=["a_code", "fy_be"], how="left")

def build_dataset(mode=EXPOSURE_MODE, lag=LAG, outcome=OUTCOME, exclude_outliers=EXCLUDE_OUTLIERS, verbose=False):
    E = exposure_period(mode, lag)
    D = E.merge(HDC, on=["a_code", "ym"], how="inner").merge(BND[["a_code", "name", "pop"]], on="a_code", how="inner")
    D["y"] = outcome_values(D, outcome).round().astype(int)
    D = D[(D["pop"] > 0) & D.pm_use.notna()].copy()
    if COV is not None:
        D = D.merge(COV, left_on="a_code", right_index=True, how="inner").dropna(subset=list(COV.columns))
    # อัตราตำบลเทียบจังหวัด (ทั้งช่วง)
    g = D.groupby("a_code").agg(cases=("y", "sum"), n=("y", "size"), pop=("pop", "first"))
    g["ratio"] = (g.cases / (g["pop"] * g.n)) / (D.y.sum() / (D["pop"].sum()))
    out = g.index[g.ratio >= OUTLIER_RATIO]
    if exclude_outliers: D = D[~D.a_code.isin(out)].copy()
    # การสัมผัส: between (ค่าเฉลี่ยของตำบล) และ within (ส่วนเบี่ยงเบนรายเดือน) ต่อ PM_UNIT
    D["pm_mean_i"] = D.groupby("a_code")["pm_use"].transform("mean")
    D["pm_b"] = (D.pm_mean_i - D.pm_mean_i.mean()) / PM_UNIT
    D["pm_w"] = (D.pm_use - D.pm_mean_i) / PM_UNIT
    D["pm_c"] = (D.pm_use - D.pm_use.mean()) / PM_UNIT
    if TEMPORAL == "annual":
        D["moy"] = 1; D["fy"] = D.ym.str[1:].astype(int)
    else:
        t = pd.to_datetime(D.ym)
        D["moy"] = t.dt.month; D["fy"] = t.dt.year + 543 + (t.dt.month >= 10).astype(int)
    D["log_pop"] = np.log(D["pop"])
    if verbose:
        print(f"{len(D):,} แถว = {D.a_code.nunique()} ตำบล × {D.ym.nunique()} {UNIT} ({D.ym.min()} ถึง {D.ym.max()}) | "
              f"OUTCOME = {outcome}, ผู้ป่วยรวม {D.y.sum():,} | PM2.5 = {mode}, lag {lag}")
    return D, g.sort_values("ratio", ascending=False), list(out)

D, RATIO, OUTLIERS = build_dataset(verbose=True)
print(f"PM2.5 ที่ใช้: เฉลี่ย {D.pm_use.mean():.1f} µg/m³ | SD ระหว่างตำบล {D.groupby('a_code').pm_use.mean().std():.2f} | "
      f"SD ภายในตำบล (ราย{UNIT}) {(D.pm_w * PM_UNIT).std():.2f} µg/m³")
print(f"ค่าเฉลี่ย {D.y.mean():.1f}, ความแปรปรวน {D.y.var():.0f} → variance/mean = {D.y.var() / D.y.mean():.0f} (≫1 = overdispersion)")
print(f"\nตำบลที่อัตราป่วย ≥ {OUTLIER_RATIO} เท่าของจังหวัด: {len(OUTLIERS)} ตำบล" + (" (ตัดออกจากแบบจำลองหลัก)" if EXCLUDE_OUTLIERS else " (ยังอยู่ในแบบจำลองหลัก)"))
display(RATIO.join(BND.set_index("a_code")["name"]).head(12).round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.3))
m_ = D.groupby("ym").agg(rate=("y", "sum"), pop=("pop", "sum"), pm=("pm_use", "mean"))
m_.index = pd.to_datetime(m_.index) if TEMPORAL == "monthly" else m_.index.str[1:].astype(int)
ax = axes[0]; ax.plot(m_.index, m_.rate / m_["pop"] * 1000, "-o", ms=3, color="#7a3b8f", label=f"{OUTCOME} ต่อ 1,000 คน/{UNIT}")
ax2 = ax.twinx(); ax2.plot(m_.index, m_.pm, "-o" if TEMPORAL == "annual" else "-", color="#e08214", label="PM2.5 (µg/m³)"); ax2.grid(False)
ax.axvline(pd.Timestamp("2025-01-01") if TEMPORAL == "monthly" else 2567.5, ls="--", c="gray")
ax.set_title(f"ค่าเฉลี่ยจังหวัดราย{UNIT} (เส้นประ = เริ่มใช้สมการที่ 15)")
ax.legend(loc="upper left", fontsize=8); ax2.legend(loc="upper right", fontsize=8)
lr_ = np.log((D.y + 0.5) / D["pop"])
axes[1].scatter(D.pm_w * PM_UNIT, lr_ - lr_.groupby(D.a_code).transform("mean"), s=3, alpha=.25)
axes[1].set_xlabel("PM2.5 ภายในตำบล: ส่วนเบี่ยงเบนจากค่าเฉลี่ยของตำบล (µg/m³)")
axes[1].set_ylabel("log อัตราป่วย − log ค่าเฉลี่ยของตำบล"); axes[1].set_title("ความสัมพันธ์ภายในตำบล (ก่อนควบคุมฤดูกาล)")
plt.tight_layout(); plt.show()

## 4. แบบจำลอง NB แบบ fixed effect รายตำบล (เร็ว) และการวิเคราะห์ความไว

$$\ln\mu_{it}=\ln POP_i+\alpha_i+\beta_W\,PM^{W}_{it}+\text{เวลา}_t$$

$\alpha_i$ = fixed effect ของตำบล (ตัดทุกปัจจัยที่คงที่ของตำบลออก) → $e^{\beta_W}$ คือ IRR แบบ *within* ต่อ 10 µg/m³
ใช้ตรวจว่าผลหลักไวต่อทางเลือกในการวิเคราะห์หรือไม่ (แต่ละแบบใช้เวลาไม่กี่วินาที) — ผลหลักสำหรับรายงานคือแบบ Bayesian ในส่วนที่ 6–8

In [ ]:
def time_terms(tc):
    if TEMPORAL == "annual": return ["C(fy)"]
    return ["C(moy)", "C(fy)"] if tc == "moy+fy" else ["C(ym)"]

def fit_freq(D_, tc=TIME_CONTROL):
    f = "y ~ pm_w + C(a_code) + " + " + ".join(time_terms(tc))
    try:
        r = smf.negativebinomial(f, data=D_, offset=D_.log_pop).fit(disp=0, maxiter=400, method="bfgs")
        b, se = r.params["pm_w"], r.bse["pm_w"]; alpha = r.params.get("alpha", np.nan)
        if not np.isfinite(se): raise RuntimeError("SE")
    except Exception:
        p0 = smf.glm(f, data=D_, family=sm.families.Poisson(), offset=D_.log_pop).fit()
        alpha = max(((D_.y - p0.mu) ** 2 - p0.mu).sum() / (p0.mu ** 2).sum(), 1e-3)
        r = smf.glm(f, data=D_, family=sm.families.NegativeBinomial(alpha=alpha), offset=D_.log_pop).fit()
        b, se = r.params["pm_w"], r.bse["pm_w"]
    z = 1.959964
    return {"IRR": np.exp(b), "lo": np.exp(b - z * se), "hi": np.exp(b + z * se), "p": 2 * (1 - __import__("scipy").stats.norm.cdf(abs(b / se))),
            "n_rows": len(D_), "n_tambon": D_.a_code.nunique(), "n_month": D_.ym.nunique(), "alpha": alpha}

SCEN = [("หลัก", dict()),
        (f"lag {1 - LAG} {UNIT}", dict(lag=1 - LAG)),
        ("PM2.5 ตามโครงร่าง (สเกลสถานี)", dict(mode="proposal")),
        ("เฉพาะเดือนที่มีดาวเทียม", dict(mode="satellite_only")),
        (f"{'ตัด' if not EXCLUDE_OUTLIERS else 'ไม่ตัด'}ตำบลอัตราผิดปกติ", dict(exclude_outliers=not EXCLUDE_OUTLIERS)),
        ] + ([("ควบคุมเวลาเข้ม (ผลทุกเดือน)", dict(_tc="month_fe" if TIME_CONTROL == "moy+fy" else "moy+fy"))] if TEMPORAL == "monthly" else [])
SCEN += [(f"กลุ่มโรค: {o}", dict(outcome=o)) for o in GROUPS if o != OUTCOME]

rows = []
for label, kw in SCEN:
    tc = kw.pop("_tc", TIME_CONTROL)
    t0 = time.time()
    try:
        D_, _, _ = build_dataset(**kw)
        r = fit_freq(D_, tc); r.update(scenario=label, time_control=tc)
        rows.append(r); print(f"✓ {label}: IRR {r['IRR']:.3f} ({r['lo']:.3f}–{r['hi']:.3f}) [{time.time() - t0:.0f} s]", flush=True)
    except Exception as e:
        print(f"✗ {label}: {type(e).__name__}: {str(e)[:100]}")
SENS = pd.DataFrame(rows).set_index("scenario")
SENS["%change_per10"] = (SENS.IRR - 1) * 100
display(SENS[["IRR", "lo", "hi", "p", "%change_per10", "n_tambon", "n_month", "n_rows", "time_control"]].round(4))

fig, ax = plt.subplots(figsize=(7.5, 0.42 * len(SENS) + 1))
yy = np.arange(len(SENS))[::-1]
ax.errorbar(SENS.IRR, yy, xerr=[SENS.IRR - SENS.lo, SENS.hi - SENS.IRR], fmt="o", capsize=3, color="#2c7bb6")
ax.set_yticks(yy, SENS.index); ax.axvline(1, ls="--", c="k")
ax.set_xlabel(f"IRR ต่อ PM2.5 ภายในตำบล {PM_UNIT} µg/m³ (95% CI)"); ax.set_title("NB fixed effect รายตำบล — การวิเคราะห์ความไว", fontsize=10)
plt.tight_layout(); plt.show()

## 5. โครงข่ายเพื่อนบ้านและ scaling factor ของ BYM2
- Queen contiguity บนตำบลที่อยู่ในแบบจำลอง · ถ้ามีส่วนที่แยกกัน (เกาะ) จะเชื่อมกับตำบลที่ใกล้ที่สุดอัตโนมัติ
- Scaling factor $s$ = ค่าเฉลี่ยเรขาคณิตของความแปรปรวนขอบของ ICAR (Riebler et al., 2016)

In [ ]:
TAM = BND[BND.a_code.isin(D.a_code.unique())].reset_index(drop=True)
TIDX = {c: i for i, c in enumerate(TAM.a_code)}; N = len(TAM)
D["tidx"] = D.a_code.map(TIDX).astype(int)
W = Queen.from_dataframe(TAM, use_index=False, silence_warnings=True)
A = W.sparse.toarray().astype(float); A = ((A + A.T) > 0).astype(float)
cen = np.c_[TAM.to_crs(UTM).centroid.x, TAM.to_crs(UTM).centroid.y]
ncomp, lab = connected_components(sparse.csr_matrix(A), directed=False)
while ncomp > 1:
    i_in, i_out = np.where(lab == 0)[0], np.where(lab != 0)[0]
    d = np.sqrt(((cen[i_in, None, :] - cen[None, i_out, :]) ** 2).sum(-1)); a, b = np.unravel_index(d.argmin(), d.shape)
    A[i_in[a], i_out[b]] = A[i_out[b], i_in[a]] = 1
    ncomp, lab = connected_components(sparse.csr_matrix(A), directed=False)

def bym2_scaling(A):
    Q = np.diag(A.sum(1)) - A
    Qp = Q + np.eye(len(A)) * Q.diagonal().max() * np.sqrt(np.finfo(float).eps)
    Qi = np.linalg.inv(Qp); one = np.ones((1, len(A)))
    Qc = Qi - Qi @ one.T @ np.linalg.inv(one @ Qi @ one.T) @ one @ Qi
    return float(np.exp(np.mean(np.log(np.diag(Qc)))))
def icar_cov(A):
    """generalised inverse ของ precision ICAR ภายใต้ผลรวม = 0"""
    Q = np.diag(A.sum(1)) - A
    Qp = Q + np.eye(len(A)) * Q.diagonal().max() * np.sqrt(np.finfo(float).eps)
    Qi = np.linalg.inv(Qp); one = np.ones((1, len(A)))
    return Qi - Qi @ one.T @ np.linalg.inv(one @ Qi @ one.T) @ one @ Qi
SCALE = bym2_scaling(A)
S_ICAR = icar_cov(A) / SCALE            # ความแปรปรวนร่วมของ ICAR ที่ scale แล้ว (ค่าเฉลี่ยเรขาคณิตของ diagonal = 1)
print(f"{N} ตำบล, เพื่อนบ้านเฉลี่ย {A.sum(1).mean():.2f}, connected components = {ncomp}, BYM2 scaling factor s = {SCALE:.3f}")

# Moran's I ของอัตราป่วยเฉลี่ยรายตำบล (log) — มีโครงสร้างเชิงพื้นที่หรือไม่
rate_i = np.log(D.groupby("tidx").apply(lambda g: g.y.sum() / (g["pop"].iloc[0] * len(g))).reindex(range(N)).values)
Wr = libpysal.weights.full2W(A); Wr.transform = "r"
MI = esda.Moran(rate_i, Wr, permutations=999)
print(f"Moran's I ของ log อัตราป่วยรายตำบล = {MI.I:.3f}, p = {MI.p_sim:.3f}")

## 6. แบบจำลอง Bayesian

**NB-iid** (ผลสุ่มรายตำบลแบบไม่มีโครงสร้าง) และ **NB-BYM2**

$$y_{it}\sim\text{NegBin}(\mu_{it},\alpha)$$
$$\ln\mu_{it}=\ln POP_i+\beta_0+\beta_W\,PM^{W}_{it}+\beta_B\,PM^{B}_{i}+\mathbf{x}_i^\top\boldsymbol\gamma+\delta_{m(t)}+\tau_{fy(t)}+\phi_i$$
$$\phi_i=\sigma\Big(\sqrt{\rho/s}\,u_i+\sqrt{1-\rho}\,v_i\Big),\quad u\sim\text{ICAR},\; v_i\sim N(0,1)
\;\;\Longleftrightarrow\;\; \boldsymbol\phi\sim N\!\Big(\mathbf 0,\;\sigma^2\big[(1-\rho)\mathbf I+\rho\,\mathbf Q_{*}^{-}\big]\Big)$$

- $PM^{W}_{it}=(PM_{it}-\overline{PM}_i)/10$ — ส่วนภายในตำบล (**ผลหลัก**) · $PM^{B}_i=(\overline{PM}_i-\overline{PM})/10$ — ส่วนระหว่างตำบล
- $\delta_m$ = ผลของเดือนในรอบปี (ฤดูกาล) · $\tau_{fy}$ = ผลของปีงบประมาณ (รวมผลช่วงโควิดปีงบ 2565) — หรือผลของทุกเดือนเมื่อ `TIME_CONTROL = "month_fe"`
- NB-iid คือกรณี $\rho = 0$ (ไม่มีโครงสร้างเชิงพื้นที่)
- Priors: $\beta \sim N(0,1)$, $\delta,\tau \sim$ ZeroSumNormal(1), $\sigma\sim\text{HalfNormal}(1)$, $\rho\sim\text{Beta}(0.5,0.5)$, $1/\alpha\sim\text{HalfNormal}(1)$

**การเขียนโค้ด (เพื่อให้ MCMC ลู่เข้า):** ข้อมูลรายตำบลมีผู้ป่วยหลายพันครั้ง ผลรายตำบลจึงถูกกำหนดโดยข้อมูลชัดเจน — ใช้ *hierarchical centering* (Gelfand et al., 1995):
$\theta_i=\beta_0+\beta_B PM^B_i+\mathbf x_i^\top\boldsymbol\gamma+\phi_i$ แจกแจงแบบปกติหลายตัวแปรด้วยความแปรปรวนร่วมของ BYM2 ข้างบน (รูปแบบ marginal ของ BYM2 — แบบจำลองเดียวกันทุกประการ)
และผลของเวลาแบบผลรวมเป็นศูนย์ · NumPyro ใช้ dense mass matrix (ทดสอบกับข้อมูลจำลอง: ESS > 600 ทุกพารามิเตอร์ด้วย 2 chains × 400 draws)

In [ ]:
y_obs = D.y.values; log_off = D.log_pop.values; tidx = D.tidx.values
x_w, x_b = D.pm_w.values, D.pm_b.values
X_cols = []
if COV is not None:
    for v in COV.columns:
        x = np.log1p(D[v].clip(lower=0))
        if x.std() > 0: D["z_" + v] = (x - x.mean()) / x.std(); X_cols.append("z_" + v)
Xmat = D[X_cols].values if X_cols else None
if TEMPORAL == "annual":
    T1 = pd.Categorical(D.fy).codes.astype(int); n1 = int(T1.max() + 1); T2 = None
elif TIME_CONTROL == "moy+fy":
    T1 = pd.Categorical(D.moy).codes.astype(int); T2 = pd.Categorical(D.fy).codes.astype(int); n1, n2 = int(T1.max() + 1), int(T2.max() + 1)
else:
    T1 = pd.Categorical(D.ym).codes.astype(int); n1 = int(T1.max() + 1); T2 = None
b0_center = float(np.log(D.y.sum() / D["pop"].sum()))

XI = D.groupby("tidx")[X_cols].first().reindex(range(N)).values if X_cols else None    # ตัวแปรระดับตำบล
xb_i = D.groupby("tidx")["pm_b"].first().reindex(range(N)).values
EYE = np.eye(N)

def build(spatial=True):
    with pm.Model() as m:
        b0 = pm.Normal("b0", b0_center, 2)
        b_w = pm.Normal("b_within", 0, 1); b_b = pm.Normal("b_between", 0, 1)
        mu_i = b0 + b_b * xb_i
        if X_cols:
            g = pm.Normal("gamma", 0, 1, shape=len(X_cols)); mu_i = mu_i + pt.dot(XI, g)
        sigma = pm.HalfNormal("sigma", 1)
        if spatial:
            rho = pm.Beta("rho", 0.5, 0.5)
            cov = sigma ** 2 * ((1 - rho) * EYE + rho * S_ICAR) + 1e-6 * EYE
            theta = pm.MvNormal("theta", mu=mu_i, cov=cov)
        else:
            theta = pm.Normal("theta", mu_i, sigma, shape=N)
        pm.Deterministic("phi", theta - mu_i)
        eta = log_off + theta[tidx] + b_w * x_w
        d1 = pm.ZeroSumNormal("time1", sigma=1, shape=n1); eta = eta + d1[T1]
        if T2 is not None:
            d2 = pm.ZeroSumNormal("time2", sigma=1, shape=n2); eta = eta + d2[T2]
        inv_a = pm.HalfNormal("inv_alpha", 1)
        pm.NegativeBinomial("y", mu=pt.exp(eta), alpha=1 / inv_a, observed=y_obs)
    return m

def pick_sampler():
    if SAMPLER != "auto": return SAMPLER
    return "numpyro" if importlib.util.find_spec("numpyro") and importlib.util.find_spec("jax") else "pymc"

FITS = {}
for name, sp in [("NB-iid", False), ("NB-BYM2", True)]:
    smp = pick_sampler(); t0 = time.time(); print(f"\n=== {name} ({smp}, {CHAINS} chains × {DRAWS} draws, tune {TUNE}) ===", flush=True)
    with build(sp):
        try:
            kw = dict(nuts_sampler_kwargs={"nuts_kwargs": {"dense_mass": True}}) if smp == "numpyro" else dict(init="jitter+adapt_full")
            FITS[name] = pm.sample(DRAWS, tune=TUNE, chains=CHAINS, target_accept=0.9, random_seed=SEED, nuts_sampler=smp,
                                   progressbar=False, idata_kwargs={"log_likelihood": True}, **kw)
        except Exception as e:
            if smp == "pymc": raise
            print(f"   {smp} ใช้ไม่ได้ ({type(e).__name__}: {str(e)[:80]}) → ใช้ sampler ของ PyMC")
            FITS[name] = pm.sample(DRAWS, tune=TUNE, chains=CHAINS, target_accept=0.9, random_seed=SEED, init="jitter+adapt_full",
                                   progressbar=False, idata_kwargs={"log_likelihood": True})
    print(f"เสร็จใน {time.time() - t0:.0f} s")

## 7. ตรวจการลู่เข้าของ MCMC (ต้องผ่านก่อนแปลผล)
เกณฑ์: $\hat R$ ≤ 1.01 · ESS bulk และ tail > 400 · divergences = 0 — ถ้าไม่ผ่าน ให้เพิ่ม `TUNE`/`DRAWS` แล้วรันส่วนที่ 6 ใหม่

In [ ]:
CONV = {}
for name, idata in FITS.items():
    vars_ = ["b0", "b_within", "b_between", "sigma", "inv_alpha"] + (["gamma"] if X_cols else []) + (["rho"] if "rho" in idata.posterior else [])
    s = az.summary(idata, var_names=vars_)
    div = int(idata.sample_stats["diverging"].sum())
    ok = (s["r_hat"].max() <= 1.01) and (s["ess_bulk"].min() > 400) and (s["ess_tail"].min() > 400) and div == 0
    CONV[name] = dict(max_rhat=s["r_hat"].max(), min_ess_bulk=s["ess_bulk"].min(), min_ess_tail=s["ess_tail"].min(), divergences=div, passed=ok)
    print(f"\n{name}: max R-hat = {s['r_hat'].max():.3f}, min ESS bulk = {s['ess_bulk'].min():.0f}, "
          f"min ESS tail = {s['ess_tail'].min():.0f}, divergences = {div}  →  {'ผ่าน ✓' if ok else 'ไม่ผ่าน ✗ (ห้ามแปลผล)'}")
    display(s.round(3))
try:
    az.plot_trace(FITS["NB-BYM2"], var_names=["b_within", "b_between", "sigma", "rho"]); plt.tight_layout(); plt.show()
except Exception as e:
    print("plot_trace:", str(e)[:80])

## 8. ผลลัพธ์: IRR ต่อ PM2.5 10 µg/m³ (95% credible interval)
- **within** = ผลหลัก: เมื่อ PM2.5 ของตำบลสูงกว่าระดับปกติของตำบลนั้น 10 µg/m³ (หลังควบคุมฤดูกาลและปีงบ) จำนวนผู้ป่วยเปลี่ยนไปกี่เท่า
- **between** = ตำบลที่ค่าเฉลี่ยระยะยาวสูงกว่า 10 µg/m³ เสี่ยงต่างกันเท่าไร — ตีความระวัง (ecological/spatial confounding)
- **P(IRR > 1)** = ความน่าจะเป็นภายหลังว่าความสัมพันธ์เป็นบวก

In [ ]:
def irr_rows(name, idata):
    out = []
    terms = [("b_within", "PM2.5 within (ต่อ 10 µg/m³)"), ("b_between", "PM2.5 between (ต่อ 10 µg/m³)")]
    post = idata.posterior
    for v, lab in terms:
        e = np.exp(post[v].values.ravel()); out.append((name, lab, e))
    if X_cols:
        g = post["gamma"].values.reshape(-1, len(X_cols))
        for k, c in enumerate(X_cols): out.append((name, f"{c[2:]} (ต่อ 1 SD)", np.exp(g[:, k])))
    return [{"model": m, "term": t, "IRR": np.median(e), "CrI_lo": np.quantile(e, (1 - CONF) / 2),
             "CrI_hi": np.quantile(e, 1 - (1 - CONF) / 2), "P(IRR>1)": (e > 1).mean(), "%change": (np.median(e) - 1) * 100} for m, t, e in out]

RES = pd.DataFrame([r for n, idt in FITS.items() for r in irr_rows(n, idt)])
fr = SENS.loc["หลัก"]
RES = pd.concat([RES, pd.DataFrame([{"model": "NB fixed effect (freq.)", "term": "PM2.5 within (ต่อ 10 µg/m³)", "IRR": fr.IRR,
                                     "CrI_lo": fr.lo, "CrI_hi": fr.hi, "P(IRR>1)": np.nan, "%change": (fr.IRR - 1) * 100}])], ignore_index=True)
display(RES.round(4))

post = FITS["NB-BYM2"].posterior
print(f"σ (SD ผลสุ่มรายตำบล) = {post['sigma'].median().item():.3f} | ρ (สัดส่วนเชิงพื้นที่) = {post['rho'].median().item():.2f} "
      f"[{post['rho'].quantile(0.025).item():.2f}, {post['rho'].quantile(0.975).item():.2f}]")

t_ = RES[RES.term.str.startswith("PM2.5")]
fig, ax = plt.subplots(figsize=(8, 3.6))
cols = {"NB-iid": "C0", "NB-BYM2": "C3", "NB fixed effect (freq.)": "grey"}
for k, (mname, g) in enumerate(t_.groupby("model", sort=False)):
    yy = np.array([0 if "within" in t else 2 for t in g.term]) + k * 0.4
    ax.errorbar(g.IRR, yy, xerr=[g.IRR - g.CrI_lo, g.CrI_hi - g.IRR], fmt="o", capsize=3, color=cols.get(mname), label=mname)
ax.set_yticks([0.4, 2.4], ["within", "between"]); ax.axvline(1, ls="--", c="k")
ax.set_xlabel(f"IRR ต่อ PM2.5 {PM_UNIT} µg/m³ (95% interval)"); ax.legend(fontsize=8); ax.set_title(f"OUTCOME = {OUTCOME}, lag {LAG}", fontsize=10)
plt.tight_layout(); plt.show()

## 9. ความเสี่ยงคงเหลือรายตำบล (residual relative risk)
$\exp(\phi_i)$ = ความเสี่ยงของตำบล $i$ เทียบค่าเฉลี่ยจังหวัด **หลังควบคุมตัวแปรในแบบจำลอง** · Exceedance probability $P(\exp(\phi_i)>1)$ ≥ 0.9 = ตำบลที่น่าจะเสี่ยงสูงกว่าจริง
ตำบลที่ค่าสูงมาก (หลายเท่า) ให้ตรวจว่าเป็นที่ตั้งโรงพยาบาลหรือไม่ (ดูส่วนที่ 3)

In [ ]:
phi = post["phi"].values.reshape(-1, N)
TAM["RR_resid"] = np.exp(np.median(phi, 0)); TAM["P_exceed"] = (phi > 0).mean(0)
TAM["crude_ratio"] = TAM.a_code.map(RATIO["ratio"]).values
TAM["pm25_mean"] = TAM.a_code.map(D.groupby("a_code").pm_use.mean()).values
fig, axes = plt.subplots(1, 4, figsize=(22, 6))
TAM.plot("pm25_mean", ax=axes[0], cmap="YlOrRd", edgecolor="grey", lw=.3, legend=True, legend_kwds={"shrink": .6, "label": "µg/m³"})
axes[0].set_title("PM2.5 เฉลี่ยรายตำบล (ช่วงวิเคราะห์)", fontsize=9)
for ax, col, ttl in [(axes[1], "crude_ratio", "Crude rate ratio"), (axes[2], "RR_resid", "NB-BYM2 residual RR = exp(φ)")]:
    TAM.plot(col, ax=ax, cmap="RdYlBu_r", norm=plt.matplotlib.colors.LogNorm(0.3, 3), edgecolor="grey", lw=.3, legend=True, legend_kwds={"shrink": .6})
    ax.set_title(ttl, fontsize=9)
TAM.plot("P_exceed", ax=axes[3], cmap="Purples", vmin=0, vmax=1, edgecolor="grey", lw=.3, legend=True, legend_kwds={"shrink": .6})
TAM[TAM.P_exceed >= .9].boundary.plot(ax=axes[3], color="k", lw=1.4); axes[3].set_title("P(exp(φ) > 1) (เส้นดำ ≥ 0.9)", fontsize=9)
for a in axes: a.set_axis_off()
plt.tight_layout(); plt.show()
display(TAM.sort_values("RR_resid", ascending=False)[["a_code", "name", "pm25_mean", "crude_ratio", "RR_resid", "P_exceed"]].head(10).round(3))

## 10. เปรียบเทียบแบบจำลองด้วย LOO-CV
ELPD สูงกว่า = อธิบายข้อมูลใหม่ได้ดีกว่า · ต่างกันเกิน ~2 × `dse` ถือว่าชัดเจน · คำเตือน Pareto-k สูงพบได้บ่อยในแบบจำลองที่มีผลสุ่มรายพื้นที่

In [ ]:
CMP = None
try:
    try: CMP = az.compare(FITS, ic="loo")
    except TypeError: CMP = az.compare(FITS)
    display(CMP.round(2))
except Exception as e:
    print("LOO ใช้ไม่ได้:", str(e)[:150])

## 11. บันทึกผลและข้อความสำหรับบทที่ 3–4

In [ ]:
tag = f"{OUTCOME}_lag{LAG}_{EXPOSURE_MODE}"
RES.to_csv(f"{OUTDIR}/irr_{tag}.csv", index=False, encoding="utf-8-sig")
SENS.reset_index().to_csv(f"{OUTDIR}/sensitivity_{tag}.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(CONV).T.to_csv(f"{OUTDIR}/convergence_{tag}.csv", encoding="utf-8-sig")
D.drop(columns=[c for c in D.columns if c.startswith("z_")]).to_csv(f"{OUTDIR}/analysis_panel_{tag}.csv", index=False, encoding="utf-8-sig")
TAM.to_file(f"{OUTDIR}/tambon_residual_risk_{tag}.gpkg", driver="GPKG")
for nm, idata in FITS.items():
    try: idata.to_netcdf(f"{OUTDIR}/posterior_{nm}_{tag}.nc")
    except Exception as e: print("บันทึก posterior ไม่ได้:", str(e)[:80])

w = RES[(RES.model == "NB-BYM2") & RES.term.str.contains("within")].iloc[0]
bt = RES[(RES.model == "NB-BYM2") & RES.term.str.contains("between")].iloc[0]
conv_ok = CONV["NB-BYM2"]["passed"]
TXT = (f"วิธีวิจัย: จำนวนผู้ป่วยรายตำบลราย{UNIT} ({D.ym.min()} ถึง {D.ym.max()}, {D.a_code.nunique()} ตำบล, {len(D):,} ตำบล-{UNIT}) "
       f"วิเคราะห์ด้วย Bayesian negative binomial regression ที่มีผลสุ่มเชิงพื้นที่แบบ BYM2 (Riebler et al., 2016) "
       f"ใช้ log ประชากรเป็น offset แยก PM2.5 ถ่วงประชากรเป็นส่วนภายในตำบล (ส่วนเบี่ยงเบนราย{UNIT}จากค่าเฉลี่ยของตำบล) และส่วนระหว่างตำบล "
       f"ควบคุม{'ผลของปี' if TEMPORAL == 'annual' else ('ผลของเดือนในรอบปีและปีงบประมาณ' if TIME_CONTROL == 'moy+fy' else 'ผลของทุกเดือน')} ประมาณค่าด้วย NUTS "
       f"({CHAINS} chains × {DRAWS} draws) ตรวจการลู่เข้าด้วย R̂ ≤ 1.01, ESS > 400 และไม่มี divergent transitions\n\n"
       f"ผล{'' if conv_ok else ' (ยังไม่ผ่านเกณฑ์การลู่เข้า — ห้ามรายงาน)'}: PM2.5 ภายในตำบลที่เพิ่มขึ้น {PM_UNIT} µg/m³ สัมพันธ์กับ{OUTCOME} "
       f"IRR = {w.IRR:.3f} (95% CrI {w.CrI_lo:.3f}–{w.CrI_hi:.3f}; P(IRR>1) = {w['P(IRR>1)']:.2f}) "
       f"ส่วนความต่างระหว่างตำบล IRR = {bt.IRR:.3f} ({bt.CrI_lo:.3f}–{bt.CrI_hi:.3f}) "
       f"สัดส่วนความแปรปรวนเชิงพื้นที่ ρ = {post['rho'].median().item():.2f}")
print(TXT); open(f"{OUTDIR}/report_{tag}_th.txt", "w", encoding="utf-8").write(TXT)
print("\nบันทึกแล้วใน", OUTDIR)

## 12. ข้อจำกัดและการรายงาน

**ข้อจำกัดที่ต้องเขียนในบทความ**
1. **ecological design** — ความสัมพันธ์ระดับตำบล ไม่ใช่ระดับบุคคล
2. **ตัวเศษคือการมารับบริการที่บันทึกใน HDC** — ขึ้นกับการเข้าถึงบริการและการบันทึกรหัสโรค · ต้องระบุว่า `areacode` เป็นที่อยู่ผู้ป่วยหรือที่ตั้งหน่วยบริการ
3. **PM2.5 ม.ค. 2568 – ก.ย. 2569 มาจากสมการที่ (15)** อิงสถานีเพียง 2 แห่ง และสมมติว่ารูปแบบเชิงพื้นที่รายฤดูคงเดิม → ตรวจด้วยการวิเคราะห์เฉพาะเดือนที่มีดาวเทียม
4. **ความละเอียด ~1 km** ไม่แยกความเข้มข้นริมถนน (ที่ 24T ดาวเทียมต่ำกว่าสถานี ~23%) → ผลอาจเอนเข้าหา 1 (exposure misclassification)
5. **ประชากร WorldPop 2020 ค่าเดียว** ใช้ทุกเดือน · ช่วงโควิด (ปีงบ 2565) การมารับบริการโรคทางเดินหายใจลดลง — ควบคุมด้วยผลของปีงบ
6. **ไม่มีตัวแปรอุตุนิยมวิทยารายเดือน** (อุณหภูมิ ความชื้น) — ฤดูกาลถูกควบคุมด้วยผลของเดือน แต่ความผันแปรของอากาศปีต่อปีอาจกวนผล → เพิ่มได้ใน `COVARIATES` แบบรายเดือนในขั้นต่อไป

**การวิเคราะห์ที่ควรทำเพิ่ม:** แยกรายโรค (`OUTCOME = "COPD"`, `"Pneumonia"`, …) · lag 0–2 · ผลแปรตามฤดู (interaction PM × ฤดู) · ผลแปรตามประชากรริมถนน (interaction PM × ตัวแปรจาก Notebook 08) · เทียบกับ R-INLA

**เอกสารอ้างอิง**
- Besag, J., York, J., & Mollié, A. (1991). Bayesian image restoration, with two applications in spatial statistics. *Annals of the Institute of Statistical Mathematics, 43*(1), 1–20.
- Riebler, A., Sørbye, S. H., Simpson, D., & Rue, H. (2016). An intuitive Bayesian spatial model for disease mapping that accounts for scaling. *Statistical Methods in Medical Research, 25*(4), 1145–1165.
- Begg, M. D., & Parides, M. K. (2003). Separation of individual-level and cluster-level covariate effects in regression analysis of correlated data. *Statistics in Medicine, 22*(16), 2591–2602.
- Gelfand, A. E., Sahu, S. K., & Carlin, B. P. (1995). Efficient parametrisations for normal linear mixed models. *Biometrika, 82*(3), 479–488.
- Hodges, J. S., & Reich, B. J. (2010). Adding spatially-correlated errors can mess up the fixed effect you love. *The American Statistician, 64*(4), 325–334.
- Vehtari, A., Gelman, A., & Gabry, J. (2017). Practical Bayesian model evaluation using leave-one-out cross-validation and WAIC. *Statistics and Computing, 27*(5), 1413–1432.
- van Donkelaar, A., et al. (2021). Monthly global estimates of fine particulate matter and their uncertainty. *Environmental Science & Technology, 55*(22), 15287–15300.

*ตรวจเลขหน้า/DOI ของเอกสารอ้างอิงอีกครั้งก่อนใส่ในต้นฉบับ*